# Module 42: Building a RAG Pipeline — Retrieval-Augmented Generation

Build a complete RAG pipeline from scratch: embedding models, vector stores, document chunking, similarity search, prompt construction, and generation — all in pure PyTorch.

| Input | Output |
|-------|--------|
| `"How does autograd work?"` | `Top-K relevant docs + generated answer` |
| `30+ PyTorch knowledge base docs` | `Chunked, embedded, indexed vector store` |

In [ ]:
import math
import time

import torch
import torch.nn as nn
import torch.nn.functional as F

print(f"PyTorch {torch.__version__}")
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## 1. What Is RAG?

**Retrieval-Augmented Generation** combines information retrieval with text generation:

```
Traditional LLM:  Query ──────────────────> LLM ──> Answer (may hallucinate)
RAG Pipeline:     Query ──> Retrieve docs ──> LLM + Context ──> Grounded answer
```

Instead of relying solely on memorized knowledge, RAG **dynamically injects relevant documents** at inference time.

## 2. Simple Tokenizer

We use a character-level tokenizer for demonstration. Production systems use BPE/WordPiece.

In [ ]:
class SimpleTokenizer:
    """Character-level tokenizer with padding."""
    def __init__(self, vocab_size=256, max_length=128):
        self.vocab_size = vocab_size
        self.max_length = max_length
        self.pad_token_id = 0

    def encode(self, text):
        return [min(ord(c), self.vocab_size - 1) for c in text][:self.max_length]

    def encode_batch(self, texts):
        encoded = [self.encode(t) for t in texts]
        max_len = min(max(len(e) for e in encoded), self.max_length)
        padded, masks = [], []
        for enc in encoded:
            length = min(len(enc), max_len)
            pad_len = max_len - length
            padded.append(enc[:length] + [0] * pad_len)
            masks.append([1] * length + [0] * pad_len)
        return torch.tensor(padded, dtype=torch.long), torch.tensor(masks, dtype=torch.long)

tokenizer = SimpleTokenizer(vocab_size=256, max_length=64)

# Quick test
ids, mask = tokenizer.encode_batch(["Hello PyTorch", "Hi"])
print(f"Token IDs shape: {ids.shape}")
print(f"Mask shape: {mask.shape}")
print(f"Mask: {mask}")

## 3. Embedding Model

A transformer encoder that produces fixed-size embeddings via **mean pooling** + **L2 normalization**.

```
Tokens -> Token Embed -> Pos Embed -> Encoder Layers -> Mean Pool -> Normalize -> Embedding
```

In [ ]:
class EmbeddingModel(nn.Module):
    """Transformer encoder for text embeddings."""
    def __init__(self, vocab_size=256, d_model=128, nhead=4, num_layers=2,
                 dim_feedforward=256, max_seq_len=128, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.position_embedding = nn.Embedding(max_seq_len, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.layer_norm = nn.LayerNorm(d_model)
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, input_ids, attention_mask=None):
        B, T = input_ids.shape
        positions = torch.arange(T, device=input_ids.device).unsqueeze(0)
        x = self.token_embedding(input_ids) * math.sqrt(self.d_model)
        x = x + self.position_embedding(positions)
        src_key_padding_mask = (attention_mask == 0) if attention_mask is not None else None
        hidden = self.encoder(x, src_key_padding_mask=src_key_padding_mask)
        hidden = self.layer_norm(hidden)
        # Mean pooling
        if attention_mask is not None:
            mask = attention_mask.unsqueeze(-1).float()
            pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
        else:
            pooled = hidden.mean(dim=1)
        return F.normalize(pooled, p=2, dim=-1)

emb_model = EmbeddingModel(vocab_size=256, d_model=128, nhead=4, num_layers=2, max_seq_len=64)
emb_model.to(device).eval()
print(f"Embedding model: {sum(p.numel() for p in emb_model.parameters()):,} params")
print(f"Embedding dimension: {emb_model.d_model}")

## 4. Mean Pooling — Step by Step

Mean pooling averages all token embeddings (excluding padding) into a single vector.

In [ ]:
# Demonstrate mean pooling step by step
texts = ["PyTorch autograd", "Tensors on GPU"]
input_ids, attention_mask = tokenizer.encode_batch(texts)
input_ids, attention_mask = input_ids.to(device), attention_mask.to(device)

with torch.no_grad():
    # Get raw encoder output
    B, T = input_ids.shape
    positions = torch.arange(T, device=device).unsqueeze(0)
    x = emb_model.token_embedding(input_ids) * math.sqrt(emb_model.d_model)
    x = x + emb_model.position_embedding(positions)
    hidden = emb_model.encoder(x)
    print(f"Hidden states shape: {hidden.shape}  (batch, seq_len, d_model)")

    # Mean pool with mask
    mask = attention_mask.unsqueeze(-1).float()
    print(f"Mask shape: {mask.shape}  (batch, seq_len, 1)")
    summed = (hidden * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    pooled = summed / counts
    print(f"Pooled shape: {pooled.shape}  (batch, d_model)")

    # L2 normalize
    normalized = F.normalize(pooled, p=2, dim=-1)
    norms = normalized.norm(dim=1)
    print(f"Norms after normalization: {norms.tolist()} (should be ~1.0)")

## 5. Text Encoder Wrapper

A high-level interface that handles tokenization, batching, and encoding.

In [ ]:
class TextEncoder:
    """Encode texts into embedding vectors."""
    def __init__(self, model, tokenizer, device):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device
        self.model.eval()

    @torch.no_grad()
    def encode(self, texts, batch_size=32):
        all_embs = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            ids, mask = self.tokenizer.encode_batch(batch)
            embs = self.model(ids.to(self.device), mask.to(self.device))
            all_embs.append(embs.cpu())
        return torch.cat(all_embs, dim=0)

    @torch.no_grad()
    def encode_single(self, text):
        return self.encode([text])[0]

encoder = TextEncoder(emb_model, tokenizer, device)
print("TextEncoder ready")

## 6. Cosine Similarity

With L2-normalized embeddings, cosine similarity = dot product:

$$\text{cos}(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \|\mathbf{b}\|} = \mathbf{a} \cdot \mathbf{b} \quad \text{when } \|\mathbf{a}\| = \|\mathbf{b}\| = 1$$

In [ ]:
texts = [
    "PyTorch autograd computes gradients",
    "Automatic differentiation engine",
    "CUDA enables GPU computation",
    "Data loading with DataLoader",
]

embeddings = encoder.encode(texts)
print(f"Embeddings shape: {embeddings.shape}")

# Pairwise similarity matrix
sim_matrix = embeddings @ embeddings.T
print(f"\nSimilarity matrix:")
labels = ["autograd", "autodiff", "CUDA", "DataLoader"]
print("            " + "  ".join(f"{l:>10}" for l in labels))
for i in range(4):
    row = f"{labels[i]:>10}  " + "  ".join(f"{sim_matrix[i,j]:10.4f}" for j in range(4))
    print(row)

# Verify dot product == cosine similarity
dot = (embeddings[0] @ embeddings[1]).item()
cos = F.cosine_similarity(embeddings[0:1], embeddings[1:2]).item()
print(f"\nDot product: {dot:.6f}")
print(f"Cosine sim:  {cos:.6f}")
print(f"Match: {abs(dot - cos) < 1e-5}")

## 7. Document Chunking Strategies

Documents are split into smaller chunks for more precise retrieval. We implement three strategies:
1. **Fixed-size**: Split by word count with overlap
2. **Sentence-based**: Split at sentence boundaries
3. **Overlapping windows**: Sliding window with configurable stride

In [ ]:
def chunk_fixed_size(text, chunk_size=50, overlap=10):
    words = text.split()
    if len(words) <= chunk_size:
        return [text]
    chunks, stride = [], max(1, chunk_size - overlap)
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunks.append(" ".join(words[start:end]))
        if end >= len(words): break
        start += stride
    return chunks

def chunk_by_sentences(text, max_words=50):
    for d in ["! ", "? "]: text = text.replace(d, ". ")
    sentences = [s.strip() for s in text.split(". ") if s.strip()]
    chunks, current, count = [], [], 0
    for sent in sentences:
        n = len(sent.split())
        if count + n > max_words and current:
            chunks.append(". ".join(current) + ".")
            current, count = [], 0
        current.append(sent)
        count += n
    if current: chunks.append(". ".join(current) + ".")
    return chunks

def chunk_overlapping(text, chunk_size=50, stride=25):
    words = text.split()
    if len(words) <= chunk_size:
        return [text]
    chunks, start = [], 0
    while start < len(words):
        chunks.append(" ".join(words[start:start + chunk_size]))
        if start + chunk_size >= len(words): break
        start += stride
    return chunks

# Compare strategies on sample text
sample = (
    "PyTorch autograd automatically computes gradients. "
    "It builds a dynamic computation graph during forward. "
    "Each operation is recorded as a node in the graph. "
    "When backward is called gradients flow in reverse. "
    "This enables automatic differentiation for any computation."
)

for name, fn, kw in [
    ("Fixed", chunk_fixed_size, dict(chunk_size=15, overlap=3)),
    ("Sentence", chunk_by_sentences, dict(max_words=15)),
    ("Overlapping", chunk_overlapping, dict(chunk_size=15, stride=8)),
]:
    chunks = fn(sample, **kw)
    print(f"\n{name} ({len(chunks)} chunks):")
    for i, c in enumerate(chunks):
        print(f"  {i}: [{len(c.split())} words] {c}")

## 8. Vector Store

The vector store holds document chunks with their embeddings and supports cosine similarity search.

In [ ]:
class VectorStore:
    def __init__(self, encoder, chunk_size=30, chunk_overlap=5):
        self.encoder = encoder
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.documents = []
        self.chunks = []
        self.embeddings = None
        self.chunk_to_doc = []

    def add_documents(self, texts):
        doc_start = len(self.documents)
        new_chunks = []
        for i, text in enumerate(texts):
            doc_chunks = chunk_fixed_size(text, self.chunk_size, self.chunk_overlap)
            for chunk in doc_chunks:
                new_chunks.append(chunk)
                self.chunk_to_doc.append(doc_start + i)
            self.documents.append(text)
        new_embs = self.encoder.encode(new_chunks)
        self.embeddings = new_embs if self.embeddings is None else torch.cat([self.embeddings, new_embs])
        self.chunks.extend(new_chunks)

    def search(self, query, k=5):
        if self.embeddings is None:
            return []
        q_emb = self.encoder.encode([query])
        sims = (q_emb @ self.embeddings.T).squeeze(0)
        k = min(k, len(self.chunks))
        top_k = torch.topk(sims, k)
        return [(self.chunks[i], top_k.values[j].item(), self.chunk_to_doc[i])
                for j, i in enumerate(top_k.indices.tolist())]

    def search_dedup(self, query, k=5):
        """One result per source document."""
        q_emb = self.encoder.encode([query])
        sims = (q_emb @ self.embeddings.T).squeeze(0)
        results, seen = [], set()
        for idx in sims.argsort(descending=True):
            idx = idx.item()
            doc_id = self.chunk_to_doc[idx]
            if doc_id in seen: continue
            seen.add(doc_id)
            results.append((self.chunks[idx], sims[idx].item(), doc_id))
            if len(results) >= k: break
        return results

store = VectorStore(encoder, chunk_size=30, chunk_overlap=5)
print("VectorStore ready")

## 9. Knowledge Base

We create a synthetic knowledge base of 30+ PyTorch documentation snippets.

In [ ]:
KNOWLEDGE_BASE = [
    "PyTorch tensors are multi-dimensional arrays with GPU support. They support automatic "
    "differentiation through autograd. Create tensors with torch.tensor, torch.zeros, torch.randn.",

    "Autograd is PyTorch's automatic differentiation engine. It records operations to build a "
    "dynamic computation graph. Call backward() to compute gradients for all requires_grad tensors.",

    "The computation graph is dynamic and rebuilt every forward pass. This allows different control "
    "flow per sample, ideal for variable-length sequences and recursive structures.",

    "nn.Module is the base class for neural networks. Subclass it and implement forward(). "
    "Key layers: nn.Linear, nn.Conv2d, nn.LSTM, nn.Embedding, nn.LayerNorm, nn.Dropout.",

    "Loss functions include nn.CrossEntropyLoss for classification, nn.MSELoss for regression, "
    "nn.BCEWithLogitsLoss for binary tasks. CrossEntropyLoss combines LogSoftmax and NLLLoss.",

    "Hooks inspect or modify module behavior. Forward hooks run after forward(), backward hooks "
    "during backprop. Register with module.register_forward_hook().",

    "Common optimizers: SGD, Adam, AdamW. AdamW decouples weight decay from gradient updates. "
    "Always call optimizer.zero_grad() before backward() and optimizer.step() after.",

    "Training loop pattern: forward pass, compute loss, backward pass, optimizer step. "
    "Use model.train() and model.eval() to switch modes. Gradient accumulation enables larger batches.",

    "Transfer learning uses pretrained models as starting points. Freeze backbone layers, "
    "train only the classifier head. Fine-tune all layers with smaller learning rate.",

    "Dataset defines sample access with __len__ and __getitem__. DataLoader handles batching, "
    "shuffling, and parallel loading with num_workers. Use collate_fn for custom batching.",

    "Data augmentation improves generalization: RandomCrop, RandomHorizontalFlip, ColorJitter, "
    "Normalize. MixUp and CutMix blend samples. Apply only during training.",

    "torch.compile optimizes models using TorchDynamo for graph capture and TorchInductor for "
    "code generation. Modes: default, reduce-overhead, max-autotune.",

    "Debugging torch.compile: use TORCH_LOGS=dynamo. Check graph breaks with torch._dynamo.explain. "
    "Common causes: data-dependent control flow, unsupported Python features.",

    "Scaled dot-product attention: Attention(Q,K,V) = softmax(QK^T/sqrt(d_k)) * V. "
    "Use torch.nn.functional.scaled_dot_product_attention for automatic backend selection.",

    "Multi-head attention splits embeddings into heads, applies attention independently, "
    "concatenates results. Each head learns different attention patterns.",

    "FlexAttention allows custom attention patterns via score_mod functions. Supports causal "
    "masking, sliding window, prefix LM. Compiled and fused for performance.",

    "DDP replicates the model on each GPU, synchronizes gradients with all-reduce after backward. "
    "Initialize with init_process_group, wrap model with DistributedDataParallel.",

    "FSDP2 shards parameters, gradients, and optimizer states across GPUs using fully_shard(). "
    "Reduces per-GPU memory compared to DDP. Uses DTensor and DeviceMesh.",

    "Pipeline Parallelism splits model into stages across GPUs. Micro-batches flow through pipeline. "
    "Strategies: GPipe, 1F1B, ZeroBubble, DualPipeV.",

    "torch.export captures models into ExportedProgram using symbolic tracing. Supports dynamic "
    "shapes via Dim. Use torch.export.export(model, args) then serialize or compile.",

    "AOTInductor compiles exported models into standalone C++ libraries. No Python needed at "
    "runtime. Suitable for production inference on CUDA and CPU targets.",

    "Functorch: vmap for vectorized map, grad for functional gradients, jacrev/jacfwd for "
    "Jacobians. Enables per-sample gradients and meta-learning.",

    "Custom operators use torch.library. Define schema, implement per backend. Add FakeTensor "
    "support for torch.compile. Works with autograd and torch.export.",

    "Mixed precision uses float16/bfloat16 to reduce memory and speed up computation. "
    "torch.amp.autocast selects precision automatically. GradScaler prevents float16 underflow.",

    "Memory optimization: gradient checkpointing, mixed precision, gradient accumulation, "
    "FSDP sharding, CPU offloading. Use torch.cuda.memory_stats to find bottlenecks.",

    "CUDA Graphs capture GPU operations and replay with minimal CPU overhead. Use with "
    "torch.cuda.CUDAGraph. torch.compile reduce-overhead mode uses CUDA Graphs automatically.",

    "RoPE encodes position by rotating query and key vectors. Preserves relative position, "
    "supports sequence length extrapolation. Used in LLaMA, Mistral, and modern LLMs.",

    "KV Cache stores computed key-value tensors during autoregressive generation, avoiding "
    "recomputation. GQA reduces cache size by sharing KV heads across query heads.",

    "LoRA adds small trainable matrices to frozen weights: W + AB where A is d x r and B is "
    "r x d with r << d. Reduces trainable parameters by 100-1000x.",

    "PyTorch testing uses TestCase with assertEqual for tensor comparison. Use @parametrize "
    "for multiple inputs. instantiate_device_type_tests for device-generic tests.",

    "Debug NaN gradients with torch.autograd.set_detect_anomaly(True). Register hooks to inspect "
    "values. Use TORCH_LOGS=dynamo for compile issues. Check with torch.isnan.",

    "torchao: post-training quantization with quantize_() for INT8, INT4, FP8. Integrates "
    "with torch.compile. 2:4 sparsity achieves roughly 2x speedup on supported GPUs.",
]

print(f"Knowledge base: {len(KNOWLEDGE_BASE)} documents")
print(f"Total words: {sum(len(d.split()) for d in KNOWLEDGE_BASE):,}")

## 10. Indexing the Knowledge Base

Chunk all documents and compute their embeddings.

In [ ]:
start = time.perf_counter()
store.add_documents(KNOWLEDGE_BASE)
elapsed = time.perf_counter() - start

print(f"Indexed {len(store.documents)} docs -> {len(store.chunks)} chunks in {elapsed:.3f}s")
print(f"Embedding matrix shape: {store.embeddings.shape}")

# Show chunk distribution
chunk_lens = [len(c.split()) for c in store.chunks]
print(f"Chunk sizes: min={min(chunk_lens)}, max={max(chunk_lens)}, avg={sum(chunk_lens)/len(chunk_lens):.1f}")

## 11. Retrieval: Query to Top-K Documents

Given a query, find the most relevant document chunks using cosine similarity.

In [ ]:
queries = [
    "How does autograd compute gradients?",
    "What is torch.compile?",
    "How to train on multiple GPUs?",
    "What is LoRA fine-tuning?",
    "How to debug NaN gradients?",
]

for query in queries:
    results = store.search(query, k=3)
    print(f"\nQuery: '{query}'")
    for rank, (chunk, score, doc_id) in enumerate(results):
        print(f"  #{rank+1} (sim={score:.4f}, doc={doc_id:2d}): {chunk[:70]}...")

## 12. Prompt Construction

Combine retrieved context with the user's query into a prompt for the LLM.

In [ ]:
def build_prompt(query, retrieved_chunks):
    """Construct prompt with retrieved context."""
    parts = []
    for i, (chunk, score, doc_id) in enumerate(retrieved_chunks):
        parts.append(f"[Source {i+1}] {chunk}")
    context = "\n".join(parts)
    return f"Context:\n{context}\n\nQuestion: {query}\n\nAnswer:"

# Example prompt
query = "How does autograd work?"
results = store.search(query, k=3)
prompt = build_prompt(query, results)
print(f"Prompt ({len(prompt)} chars):")
print(prompt)

## 13. Mini-LLM Generator

A small transformer decoder for text generation. In production, replace with a full-scale LLM.

In [ ]:
class MiniLM(nn.Module):
    """Small transformer decoder for generation demo."""
    def __init__(self, vocab_size=256, d_model=128, nhead=4, num_layers=2,
                 dim_feedforward=256, max_seq_len=512):
        super().__init__()
        self.d_model = d_model
        self.max_seq_len = max_seq_len
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=0.1, batch_first=True, norm_first=True)
        self.decoder = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        for p in self.parameters():
            if p.dim() > 1: nn.init.xavier_uniform_(p)

    def forward(self, ids):
        B, T = ids.shape
        pos = torch.arange(T, device=ids.device).unsqueeze(0)
        x = self.token_emb(ids) * math.sqrt(self.d_model) + self.pos_emb(pos)
        mask = nn.Transformer.generate_square_subsequent_mask(T, device=ids.device)
        x = self.decoder(x, mask=mask, is_causal=True)
        return self.lm_head(self.ln_f(x))

    @torch.no_grad()
    def generate(self, prompt_ids, max_new=80, temperature=0.7, top_k=50):
        self.eval()
        tokens = prompt_ids.clone()
        for _ in range(max_new):
            if tokens.shape[1] >= self.max_seq_len: break
            logits = self.forward(tokens)[:, -1, :] / temperature
            if top_k > 0:
                topk_v, _ = logits.topk(top_k)
                logits[logits < topk_v[:, -1:]] = float("-inf")
            tokens = torch.cat([tokens, torch.multinomial(F.softmax(logits, -1), 1)], 1)
        return tokens

gen_model = MiniLM(vocab_size=256, d_model=128, nhead=4, num_layers=2, max_seq_len=512).to(device)
print(f"Generator: {sum(p.numel() for p in gen_model.parameters()):,} params")

## 14. Generator Tokenizer

In [ ]:
class GenTokenizer:
    def __init__(self, vocab_size=256, max_length=512):
        self.vocab_size = vocab_size
        self.max_length = max_length

    def encode(self, text):
        ids = [min(ord(c), self.vocab_size - 1) for c in text]
        return torch.tensor([ids[:self.max_length]], dtype=torch.long)

    def decode(self, token_ids):
        if isinstance(token_ids, torch.Tensor):
            token_ids = token_ids.squeeze(0).tolist()
        return "".join(chr(min(t, 127)) for t in token_ids if 32 <= t <= 126)

gen_tok = GenTokenizer()

# Quick generation test
test_ids = gen_tok.encode("Hello ").to(device)
out_ids = gen_model.generate(test_ids, max_new=20)
print(f"Generated: '{gen_tok.decode(out_ids)}'")

## 15. RAG Pipeline

Assemble all components into a complete pipeline.

In [ ]:
class RAGPipeline:
    def __init__(self, store, gen_model, gen_tok, device):
        self.store = store
        self.gen_model = gen_model
        self.gen_tok = gen_tok
        self.device = device

    def query(self, question, top_k=3, max_tokens=60):
        # 1. Retrieve
        retrieved = self.store.search(question, k=top_k)
        # 2. Build prompt
        prompt = build_prompt(question, retrieved)
        # 3. Generate
        ids = self.gen_tok.encode(prompt).to(self.device)
        out = self.gen_model.generate(ids, max_new=max_tokens)
        answer = self.gen_tok.decode(out[:, ids.shape[1]:])
        return {"question": question, "retrieved": retrieved,
                "prompt_len": len(prompt), "answer": answer}

    def query_no_rag(self, question, max_tokens=60):
        prompt = f"Question: {question}\nAnswer:"
        ids = self.gen_tok.encode(prompt).to(self.device)
        out = self.gen_model.generate(ids, max_new=max_tokens)
        return {"question": question, "retrieved": [],
                "prompt_len": len(prompt), "answer": self.gen_tok.decode(out[:, ids.shape[1]:])}

rag = RAGPipeline(store, gen_model, gen_tok, device)
print("RAG Pipeline assembled!")

## 16. End-to-End RAG Queries

In [ ]:
rag_queries = [
    "How does autograd work?",
    "What optimizer should I use?",
    "How to speed up inference?",
    "What is LoRA?",
]

for q in rag_queries:
    result = rag.query(q, top_k=3, max_tokens=50)
    print(f"\nQ: {result['question']}")
    print(f"Retrieved {len(result['retrieved'])} chunks (prompt: {result['prompt_len']} chars)")
    for i, (chunk, score, doc_id) in enumerate(result['retrieved']):
        print(f"  Source {i+1} (sim={score:.4f}): {chunk[:60]}...")
    print(f"A: {result['answer'][:100]}")

## 17. With vs Without Retrieval

Compare RAG against a pure LLM (no context injection).

In [ ]:
comparison_queries = [
    "What is FSDP2?",
    "How does RoPE work?",
    "What is mixed precision training?",
]

for q in comparison_queries:
    with_rag = rag.query(q, top_k=3, max_tokens=40)
    without_rag = rag.query_no_rag(q, max_tokens=40)
    print(f"\nQ: {q}")
    print(f"  With RAG    (prompt={with_rag['prompt_len']:4d}): {with_rag['answer'][:80]}")
    print(f"  Without RAG (prompt={without_rag['prompt_len']:4d}): {without_rag['answer'][:80]}")

## 18. Retrieval Evaluation

Evaluate retrieval quality using keyword-based ground truth.

In [ ]:
eval_queries = [
    ("How to compute gradients?", ["autograd", "backward", "gradient"]),
    ("What is torch.compile?", ["compile", "dynamo", "inductor"]),
    ("How to train on multiple GPUs?", ["distributed", "ddp", "fsdp"]),
    ("What is LoRA?", ["lora", "low-rank", "fine-tun"]),
    ("What is attention?", ["attention", "query", "key"]),
    ("How to debug?", ["debug", "nan", "anomaly"]),
    ("What is CUDA Graphs?", ["cuda graph", "capture", "replay"]),
    ("How does KV cache work?", ["kv cache", "key", "autoregressive"]),
    ("What is quantization?", ["quantiz", "int8", "int4"]),
    ("How to export models?", ["export", "aotinductor", "deploy"]),
]

recalls, precisions, mrrs = [], [], []

for query, keywords in eval_queries:
    results = store.search(query, k=5)
    relevant_found, first_rank = 0, 0
    for rank, (chunk, score, doc_id) in enumerate(results):
        if any(kw.lower() in chunk.lower() for kw in keywords):
            relevant_found += 1
            if first_rank == 0: first_rank = rank + 1
    recalls.append(min(relevant_found / len(keywords), 1.0))
    precisions.append(relevant_found / len(results))
    mrrs.append(1.0 / first_rank if first_rank > 0 else 0.0)

print(f"Retrieval metrics over {len(eval_queries)} queries:")
print(f"  Avg Recall@5:    {sum(recalls)/len(recalls):.4f}")
print(f"  Avg Precision@5: {sum(precisions)/len(precisions):.4f}")
print(f"  Avg MRR:         {sum(mrrs)/len(mrrs):.4f}")

## 19. Latency Breakdown

Measure the time spent in each pipeline stage.

In [ ]:
query = "How does autograd compute gradients?"

t0 = time.perf_counter()
q_emb = encoder.encode([query])
t1 = time.perf_counter()
sims = (q_emb @ store.embeddings.T).squeeze(0)
top_k = sims.topk(3)
t2 = time.perf_counter()
chunks = [(store.chunks[i], top_k.values[j].item(), store.chunk_to_doc[i]) for j, i in enumerate(top_k.indices.tolist())]
prompt = build_prompt(query, chunks)
t3 = time.perf_counter()
ids = gen_tok.encode(prompt).to(device)
out = gen_model.generate(ids, max_new=50)
t4 = time.perf_counter()

print(f"Latency breakdown:")
print(f"  Encode query:      {(t1-t0)*1000:7.2f} ms")
print(f"  Similarity search: {(t2-t1)*1000:7.2f} ms")
print(f"  Build prompt:      {(t3-t2)*1000:7.2f} ms")
print(f"  Generate answer:   {(t4-t3)*1000:7.2f} ms")
print(f"  Total:             {(t4-t0)*1000:7.2f} ms")

## 20. Embedding Space Visualization

Visualize document embeddings in 2D using PCA to see how they cluster.

In [ ]:
# Simple PCA projection (no sklearn needed)
embs = store.embeddings  # (N, d_model)
centered = embs - embs.mean(dim=0)
U, S, V = torch.svd(centered)
proj_2d = (centered @ V[:, :2]).numpy()

# Group chunks by topic
topic_colors = []
for doc_id in store.chunk_to_doc:
    if doc_id < 3: topic_colors.append("Tensors/Autograd")
    elif doc_id < 6: topic_colors.append("Neural Networks")
    elif doc_id < 9: topic_colors.append("Training")
    elif doc_id < 11: topic_colors.append("Data")
    elif doc_id < 13: topic_colors.append("Compile")
    elif doc_id < 16: topic_colors.append("Attention")
    elif doc_id < 19: topic_colors.append("Distributed")
    else: topic_colors.append("Advanced")

print(f"Projected {len(proj_2d)} chunks to 2D")
print(f"Topics: {set(topic_colors)}")
print(f"\nPCA variance explained: PC1={S[0]**2/sum(S**2):.3f}, PC2={S[1]**2/sum(S**2):.3f}")

## 21. Re-Ranking Concept

Re-ranking applies a more expensive model to re-score initial retrieval candidates for better precision.

In [ ]:
def simple_rerank(query, candidates, encoder, k=3):
    """Re-rank using cross-similarity between query and each candidate.

    A simple proxy for cross-encoder re-ranking:
    encode (query + candidate) as a single text, compare to query embedding.
    """
    combined_texts = [f"{query} {chunk}" for chunk, _, _ in candidates]
    combined_embs = encoder.encode(combined_texts)
    query_emb = encoder.encode([query])
    scores = (combined_embs @ query_emb.T).squeeze(1)
    top_k = scores.topk(min(k, len(candidates)))
    return [(candidates[i][0], scores[i].item(), candidates[i][2])
            for i in top_k.indices.tolist()]

query = "How to train on multiple GPUs?"
initial = store.search(query, k=8)  # Retrieve more, then re-rank
reranked = simple_rerank(query, initial, encoder, k=3)

print(f"Query: '{query}'")
print(f"\nInitial retrieval (top 3 of 8):")
for i, (chunk, score, doc_id) in enumerate(initial[:3]):
    print(f"  #{i+1} (sim={score:.4f}): {chunk[:60]}...")

print(f"\nAfter re-ranking (top 3):")
for i, (chunk, score, doc_id) in enumerate(reranked):
    print(f"  #{i+1} (sim={score:.4f}): {chunk[:60]}...")

## 22. Query Expansion

Generate multiple search queries from a single user query for better coverage.

In [ ]:
def expand_query(query):
    """Generate multiple search queries for better recall."""
    return [
        query,
        f"What is {query.rstrip('?')}?",
        f"How to {query.rstrip('?').lower()}",
        f"Example of {query.rstrip('?').lower()}",
    ]

def multi_query_search(queries, store, k=5):
    """Search with multiple queries, deduplicate by chunk."""
    all_results = {}
    for q in queries:
        for chunk, score, doc_id in store.search(q, k=k):
            key = chunk[:50]  # Use prefix as key
            if key not in all_results or score > all_results[key][1]:
                all_results[key] = (chunk, score, doc_id)
    return sorted(all_results.values(), key=lambda x: -x[1])[:k]

query = "torch.compile"
expanded = expand_query(query)
print(f"Original: '{query}'")
print(f"Expanded: {expanded}")

single_results = store.search(query, k=3)
multi_results = multi_query_search(expanded, store, k=3)

print(f"\nSingle query results:")
for i, (chunk, score, _) in enumerate(single_results):
    print(f"  #{i+1} ({score:.4f}): {chunk[:60]}...")

print(f"\nMulti-query results:")
for i, (chunk, score, _) in enumerate(multi_results):
    print(f"  #{i+1} ({score:.4f}): {chunk[:60]}...")

## 23. Confidence Thresholding

When similarity scores are low, the retriever found nothing relevant. Handle this gracefully.

In [ ]:
def confident_search(store, query, k=3, threshold=0.3):
    """Search with confidence threshold."""
    results = store.search(query, k=k)
    confident = [(c, s, d) for c, s, d in results if s >= threshold]
    if not confident:
        return None, results  # No confident results
    return confident, results

test_queries = [
    "How does autograd work?",          # Should find relevant docs
    "What is the weather today?",       # Out of domain
    "Recipe for chocolate cake",        # Completely unrelated
]

for q in test_queries:
    confident, all_results = confident_search(store, q, k=3, threshold=0.5)
    max_score = max(s for _, s, _ in all_results) if all_results else 0
    if confident:
        print(f"\n'{q}' -> {len(confident)} confident results (max={max_score:.4f})")
    else:
        print(f"\n'{q}' -> No confident results (max={max_score:.4f})")
        print(f"  Would respond: 'I don't have enough information to answer this.'")

## 24. Persistence: Save and Load

In [ ]:
import os

save_path = "/tmp/rag_vector_store.pt"

# Save
torch.save({
    "documents": store.documents,
    "chunks": store.chunks,
    "embeddings": store.embeddings,
    "chunk_to_doc": store.chunk_to_doc,
}, save_path)

file_size = os.path.getsize(save_path)
print(f"Saved: {file_size / 1024:.1f} KB ({len(store.chunks)} chunks)")

# Load into fresh store
data = torch.load(save_path, weights_only=False)
print(f"Loaded: {len(data['chunks'])} chunks, embeddings {data['embeddings'].shape}")

# Verify search works on loaded data
loaded_sims = (encoder.encode(["autograd gradients"]) @ data["embeddings"].T).squeeze(0)
best = loaded_sims.argmax().item()
print(f"Best match: '{data['chunks'][best][:60]}...' (sim={loaded_sims[best]:.4f})")

os.remove(save_path)

## 25. Pipeline Statistics

In [ ]:
emb_params = sum(p.numel() for p in emb_model.parameters())
gen_params = sum(p.numel() for p in gen_model.parameters())

print(f"Pipeline Statistics:")
print(f"  Embedding model: {emb_params:,} params ({emb_model.d_model}-dim)")
print(f"  Generator model: {gen_params:,} params")
print(f"  Total params:    {emb_params + gen_params:,}")
print(f"  Knowledge base:  {len(store.documents)} docs, {len(store.chunks)} chunks")
print(f"  Embedding matrix: {store.embeddings.shape}")
print(f"  Memory for embeddings: {store.embeddings.nelement() * 4 / 1024:.1f} KB")

## 26. Chunk Size Experiment

How does chunk size affect retrieval quality?

In [ ]:
chunk_sizes = [10, 20, 30, 50]
test_q = "How does autograd compute gradients?"
test_keywords = ["autograd", "backward", "gradient"]

print(f"Query: '{test_q}'")
print(f"Keywords: {test_keywords}\n")

for cs in chunk_sizes:
    test_store = VectorStore(encoder, chunk_size=cs, chunk_overlap=cs // 5)
    test_store.add_documents(KNOWLEDGE_BASE)
    results = test_store.search(test_q, k=5)
    hits = sum(1 for c, s, d in results if any(kw in c.lower() for kw in test_keywords))
    top_sim = results[0][1] if results else 0
    print(f"  chunk_size={cs:2d}: {test_store.embeddings.shape[0]:3d} chunks, "
          f"{hits}/5 keyword hits, top_sim={top_sim:.4f}")

## 27. Answer Relevance Evaluation

Measure how similar generated answers are to reference answers using embedding similarity.

In [ ]:
eval_pairs = [
    ("How does autograd work?", "Autograd records operations to build a computation graph and computes gradients via backward."),
    ("What is torch.compile?", "torch.compile uses TorchDynamo and TorchInductor to optimize model execution."),
    ("What is LoRA?", "LoRA adds small trainable low-rank matrices to frozen pretrained weights."),
]

for question, reference in eval_pairs:
    result = rag.query(question, top_k=3, max_tokens=40)
    answer = result["answer"]

    ans_emb = encoder.encode([answer])
    ref_emb = encoder.encode([reference])
    similarity = (ans_emb @ ref_emb.T).item()

    print(f"\nQ: {question}")
    print(f"  Reference: {reference[:60]}...")
    print(f"  Generated: {answer[:60]}...")
    print(f"  Similarity: {similarity:.4f}")

## 28. Deduplicated Search

Get at most one result per source document for diverse results.

In [ ]:
query = "How to make training faster?"

print(f"Query: '{query}'\n")

print("Regular search (may return multiple chunks from same doc):")
for i, (chunk, score, doc_id) in enumerate(store.search(query, k=5)):
    print(f"  #{i+1} doc={doc_id:2d} (sim={score:.4f}): {chunk[:55]}...")

print("\nDeduplicated search (one per doc):")
for i, (chunk, score, doc_id) in enumerate(store.search_dedup(query, k=5)):
    print(f"  #{i+1} doc={doc_id:2d} (sim={score:.4f}): {chunk[:55]}...")

## 29. Summary

Key takeaways from building this RAG pipeline:

1. **RAG = Retrieve + Generate** — fetch relevant docs, then generate grounded answers
2. **Embedding quality is critical** — mean pooling + L2 normalization is the standard recipe
3. **Cosine similarity** — the default metric for comparing text embeddings
4. **Chunking matters** — balance chunk size, overlap, and boundary awareness
5. **Brute force is fine** — for < 100K documents, exact search is fast enough
6. **Prompt engineering** — how you format context + query affects generation quality
7. **Evaluate both stages** — measure retrieval (Recall@K) and answer quality separately
8. **Re-ranking improves precision** — two-stage retrieval combines speed with accuracy

## Exercise: Add Re-Ranking to Improve Retrieval

**Goal**: Implement a cross-encoder style re-ranker and integrate it into the RAG pipeline.

Steps:
1. Create a `CrossEncoder` class that takes (query, document) pairs and outputs a relevance score
2. Modify the RAG pipeline to retrieve top-20 with embedding search, then re-rank to top-3
3. Compare retrieval quality with and without re-ranking
4. Measure the latency overhead of re-ranking

Hint: A cross-encoder concatenates query and document as `[query] [SEP] [document]` and classifies the pair.

In [ ]:
# Exercise: Implement cross-encoder re-ranking
# Your code here!

class CrossEncoder(nn.Module):
    """Cross-encoder for re-ranking retrieved documents.

    Takes (query, document) pairs and outputs relevance scores.
    """
    def __init__(self, vocab_size=256, d_model=128, nhead=4, num_layers=2, max_seq_len=128):
        super().__init__()
        # TODO: Implement cross-encoder architecture
        # Hint: Use a transformer encoder with a classification head
        pass

    def forward(self, input_ids, attention_mask=None):
        # TODO: Return relevance scores
        pass

# TODO: Integrate into RAG pipeline
# 1. Retrieve top-20 with embedding search
# 2. Re-rank with cross-encoder to get top-3
# 3. Compare results

print("Exercise: Implement cross-encoder re-ranking")
print("See the hints above to get started!")